In [1]:
# pip install pandas
# pip install openpyxl

# Import Libraries

import pandas as pd

In [4]:
df = pd.read_excel("customer_sample_500.csv.xlsx", dtype=str)

# Replace NaN with empty string
df = df.fillna("")

print("File loaded successfully")

File loaded successfully


In [29]:
#------Completeness check function-------
def completeness_check(df, output_file="completeness_results.xlsx"):

    results = []

    # Loop through each column in the dataframe
    for col in df.columns:
        
        # Count empty or whitespace values
        missing = df[col].astype(str).str.strip().eq("").sum()

        results.append({
            "Column": col,
            "Missing Values": missing
        })

    # Convert results list into dataframe
    result_df = pd.DataFrame(results)

    # Save output to Excel
    result_df.to_excel(output_file, index=False)

    print("Output saved to", output_file)

    return result_df


# Run completeness check
result = completeness_check(df)


Output saved to completeness_results.xlsx


In [ ]:
#--uniqueness_check-----


# ---- Step 1: Find columns that should be checked ----
def find_unique_columns(df):

    unique_cols = []

    for col in df.columns:
        if df[col].nunique(dropna=False) == len(df):
            unique_cols.append(col)

    return unique_cols


unique_cols = find_unique_columns(df)


# ---- Step 2: Perform uniqueness check only on those columns ----
def uniqueness_check(df, columns, output_file="uniqueness_results.xlsx"):

    duplicate_results = []

    for col in columns:

        # Count duplicate value groups
        duplicates = (df[col].value_counts() > 1).sum()

        duplicate_results.append({
            "Column": col,
            "Duplicate_Count": duplicates
        })

    duplicate_df = pd.DataFrame(duplicate_results)

    duplicate_df.to_excel(output_file, index=False)

    print("Output saved in file:", output_file)

    return duplicate_df


# Run check only on previous result columns
result = uniqueness_check(df, unique_cols)


Output saved in file: uniqueness_results.xlsx


In [23]:
def run_validity_checks(df, output_file="Validity_results.xlsx"):

    validity_rules = {
        "Invalid_SIRET": (df["SIRET"].str.len() != 14),

        "Invalid_SIREN": (df["SIREN"].str.len() != 9),

        "Invalid_Country_Code": (df["COUNTRY_CD"].str.len() != 2),

        "Invalid_SAP_Postal_Code": (~df["SAP_POSTAL_ZIP_CD"].str.match(r'^\d+$')),

        "Invalid_LUCI_Postal_Code": (~df["LUCI_POSTAL_ZIP_CD"].str.match(r'^\d+$')),

        "Invalid_Customer_Delivery_Block":
            (~df["CUSTOMER_DELIVERY_BLOC"].isin(["00", "01"])),

        "Invalid_GERS":
            (~df["GERS"].str.match(r'^\d+$'))
    }

    delivery_columns = [
        "CUSTOMER_ORDER_BLOCK",
        "CUSTOMER_SALES_DELIVERY_BLOC",
        "CUSTOMER_SALES_ORDER_BLOCK",
        "DELETION_BLOCK"
    ]

    # Add rules dynamically for delivery columns
    for col in delivery_columns:
        validity_rules[f"{col}_Invalid"] = ~df[col].isin(["00", "01"])

    results = []

    # Count invalid rows for each rule
    for rule, condition in validity_rules.items():

        invalid_count = condition.sum()

        results.append({
            "Check": rule,
            "Invalid_Count": invalid_count
        })

    validity_df = pd.DataFrame(results)

    validity_df.to_excel(output_file, sheet_name="Validity_Check", index=False)

    print("Output saved in file:", output_file)

    return validity_df


# Run validity checks
result = run_validity_checks(df)


Output saved in file: Validity_results.xlsx


In [26]:
def run_consistency_checks(df, output_file="Consistency_results.xlsx"):

    # Dictionary defining consistency rules
    consistency_rules = {
        "Country_Mismatch": df["COUNTRY_CD"] != df["LUCI_COUNTRY_CD"]
    }

    results = []

    # Loop through rules and count mismatches
    for rule, condition in consistency_rules.items():

        mismatch_count = condition.sum()

        results.append({
            "Check": rule,
            "Mismatch_Count": mismatch_count
        })

    # Convert results to DataFrame
    consistency_df = pd.DataFrame(results)

    # Save output
    consistency_df.to_excel(output_file, sheet_name="Consistency_Check", index=False)

    print("Output saved in file:", output_file)

    return consistency_df


# Run consistency check
result = run_consistency_checks(df)


Output saved in file: Consistency_results.xlsx
